In [1]:
library(batchelor)

Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    

In [2]:
multiBatchNorm

function (..., batch = NULL, assay.type = "counts", norm.args = list(), 
    min.mean = 1, subset.row = NULL, normalize.all = FALSE, preserve.single = TRUE, 
    BPPARAM = SerialParam()) 
{
    batches <- .unpackLists(...)
    checkBatchConsistency(batches)
    if (length(batches) == 0L) {
        stop("at least one SingleCellExperiment must be supplied")
    }
    norm.args <- c(norm.args, list(exprs_values = assay.type, 
        center_size_factors = FALSE))
    needs.subset <- !normalize.all && !is.null(subset.row)
    if (.bpNotSharedOrUp(BPPARAM)) {
        bpstart(BPPARAM)
        on.exit(bpstop(BPPARAM), add = TRUE)
    }
    if (length(batches) == 1L) {
        sce <- batches[[1]]
        if (is.null(batch)) {
            stop("'batch' must be specified if '...' has only one object")
        }
        by.batch <- split(seq_along(batch), batch)
        batches <- by.batch
        for (i in seq_along(by.batch)) {
            batches[[i]] <- sce[, by.batch[[i]], drop = FALSE]
        }
    }
    else {
        preserve.single <- FALSE
    }
    sfs <- .rescale_size_factors(batches, assay.type = assay.type, 
        subset.row = subset.row, min.mean = min.mean, BPPARAM = BPPARAM)
    if (preserve.single) {
        if (needs.subset) {
            sce <- sce[subset.row, ]
        }
        all.sf <- unlist(sfs, use.names = FALSE)
        reorder <- order(unlist(by.batch))
        sizeFactors(sce) <- all.sf[reorder]
        do.call(logNormCounts, c(list(x = sce), norm.args))
    }
    else {
        for (i in seq_along(batches)) {
            current <- batches[[i]]
            sizeFactors(current) <- sfs[[i]]
            if (needs.subset) {
                current <- current[subset.row, , drop = FALSE]
            }
            current <- do.call(logNormCounts, c(list(x = current), 
                norm.args))
            batches[[i]] <- current
        }
        batches
    }
}
<bytecode: 0x563964bf9ba8>
<environment: namespace:batchelor>